> **The 45-second mystery:** Your fine-tuning loop from `02-llm-finetuning` takes 45 seconds per epoch. A colleague with "identical" hardware finishes in 8. Your first instinct is to rewrite the model. Your second instinct is to upgrade the GPU. Both are expensive and both are wrong — because you haven't measured anything yet.
>
> You add mixed precision: saves 3 seconds. You switch optimizers: nothing changes. You add `pin_memory=True` to the DataLoader: another 1 second. Two days of trial-and-error later, you're at 40 seconds — still 5× slower than your colleague.
>
> Later, a single `torch.profiler` run tells you the truth in 10 seconds: **79% of your wall time lives in the backward pass**, specifically in the layernorm gradient kernel. Your colleague's "identical" hardware has Tensor Cores enabled by a driver flag you never set. No optimizer switch, no architecture change, no data-loading trick was ever going to close that gap.
>
> The lesson is not that Tensor Cores matter (they do). The lesson is that **guessing bottlenecks costs days; measuring them costs seconds.** Every optimization technique in Chapters 4–8 makes sense only after you've measured which operation is the actual bottleneck on _your_ specific hardware.
>
> This notebook teaches you how to measure — and how to read what you find.


# PyTorch Profiling: Finding the Real Bottleneck

| Part | Tool                     | Why you need this NOW                                                                                                                                                     |
| ---- | ------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1    | `torch.profiler`         | Without operator-level timing you're guessing — this gives you the map before you navigate                                                                                |
| 2    | Profiling overhead       | Your profiler slows the code it measures; a 15% distortion can make the wrong phase look like the bottleneck                                                              |
| 3    | Compute vs. memory bound | The fix for compute-bound (Tensor Cores, bigger batch) is the _opposite_ of the fix for memory-bound (FlashAttention, fused kernels) — you must know which you're hitting |
| 4    | `torch.compile`          | One flag, potential 10–30% speedup — but only if you're compute-bound; applying it blindly wastes 60s of compilation on a workload that won't benefit                     |
| 5    | Custom profiling regions | Full profiler traces are noisy; surgical `record_function` spans isolate exactly which training sub-step is bleeding time                                                 |
| 6    | End-to-end step timing   | The Part 1 profiler sees one isolated step; a real DataLoader adds batch-loading stalls and device-transfer costs that only surface at training time                      |

---

## Prerequisite Bridge — From Ch1 GPU Hardware

| Foundation                       | Role in this notebook                                                                               |
| -------------------------------- | --------------------------------------------------------------------------------------------------- |
| Arithmetic intensity (FLOP/byte) | Determines whether profiling will show compute or bandwidth as the limit                            |
| Roofline model                   | Interprets profiler output: is the bottleneck hitting the compute ceiling or the bandwidth ceiling? |
| Memory coalescing                | Explains why some operators are slower than expected in profiles                                    |

> **If you haven't read `learning/ai-infrastructure/01-gpu-hardware/gpu-hardware-foundations.ipynb`** the compute-bound vs. memory-bound distinction used in Part 3 will be unfamiliar.


In [ ]:
import subprocess, sys

# Install any of these four packages that aren't already available
for pkg in ["torch", "numpy", "matplotlib", "transformers"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import os

# Use CUDA if a GPU is visible to PyTorch, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_GPU = torch.cuda.is_available()
torch.manual_seed(42)

print(f"Device: {DEVICE}")

# Note when running on CPU that timings are still comparable in relative terms
if HAS_GPU:
    print(f"GPU: {torch.cuda.get_device_properties(0).name}")
else:
    print("No GPU — profiling runs on CPU. Key patterns and ratios are identical.")
    print(
        "Absolute times will be slower than GPU; relative proportions are instructive."
    )

print()

#  Running example: small GPT-2-style Transformer
B, S, D = 4, 128, 256  # batch=4, seq=128, d_model=256
N_HEADS = 8
N_LAYERS = 6
VOCAB = 1000
print(
    f"Running example: Transformer (B={B}, S={S}, D={D}, {N_HEADS} heads, {N_LAYERS} layers)"
)


In [ ]:
#  Build a small Transformer for profiling
class SmallTransformer(nn.Module):
    """Small GPT-like model for profiling demonstrations."""

    def __init__(self, vocab=VOCAB, d=D, n_heads=N_HEADS, n_layers=N_LAYERS, seq=S):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        encoder_layer = nn.TransformerEncoderLayer(
            d, n_heads, dim_feedforward=d * 4, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        h = self.embed(x)
        h = self.transformer(h)
        return self.head(h)


model = SmallTransformer().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"SmallTransformer: {n_params/1e6:.1f}M parameters")

# Sample batch
x_batch = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
labels = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Warm up
for _ in range(3):
    out = model(x_batch)
    loss = criterion(out.view(-1, VOCAB), labels.view(-1))
    loss.backward()
    optimizer.zero_grad()
if HAS_GPU:
    torch.cuda.synchronize()
print("Model warmed up and ready for profiling.")

---

## Part 1 — `torch.profiler`: The Full Operator Timeline

`torch.profiler` captures every operator call during a context window, with CPU and (optionally) CUDA timing. It's the most comprehensive tool but has overhead (~5–10% slowdown).

#### #### Predict first

For one forward + backward pass through our SmallTransformer, rank the four phases by wall time:

1. **(a) data_prep > forward > backward > optimizer** — moving data dominates
2. **(b) backward > forward > optimizer > data_prep** — backward is most expensive (it computes all gradients)
3. **(c) optimizer > backward > forward > data_prep** — AdamW update is most expensive

Which order do you expect? The answer depends on your hardware and model size — that's exactly the lesson.


![Profiler timeline: data_prep (teal) + forward (amber) + backward (coral, widest) + optimizer_step (green) annotated with operator names](images/profiler-timeline-annotated.png)


### Walkthrough: kernels, launches, and the timelineBefore reading widths, name the actors. **PyTorch operations** such as `torch.matmul` are host-side API calls. A **GPU kernel** is the compiled function that thousands of GPU threads execute for one operation or part of one operation. One PyTorch operation can launch several kernels, and a fused kernel can implement several PyTorch operations.Read the annotated image from left to right:1. `data_prep` is host work and transfer setup. Its width can include allocation, copies, and CPU bookkeeping.2. `forward` launches kernels that produce activations. The Python call can finish after enqueueing work, before the GPU finishes it.3. `backward` is widest in this example because autograd launches gradient kernels for many saved operations. Width identifies where to investigate; it does not yet explain whether the cause is arithmetic, memory traffic, launch gaps, or instrumentation.4. `optimizer_step` updates parameters and optimizer state. AdamW performs several reads and writes per parameter, so it can be memory intensive even when it uses few FLOPs.```mermaidsequenceDiagram    participant CPU as CPU / Python thread    participant Q as CUDA stream queue    participant GPU as GPU    CPU->>Q: launch matmul kernel    CPU->>Q: launch softmax kernel    Note over CPU,Q: launches usually return quickly    Q->>GPU: execute matmul    GPU-->>Q: matmul complete    Q->>GPU: execute softmax    CPU->>GPU: synchronize at measurement boundary    GPU-->>CPU: all queued work complete```#### Why a normal timer can lieCUDA execution is **asynchronous** with respect to Python: the CPU usually enqueues a kernel and continues. This anti-pattern often measures launch overhead instead of GPU execution:```pythonstart = time.perf_counter()output = model(batch)                 # work is queuedelapsed = time.perf_counter() - start # GPU may still be running```For an end-to-end latency measurement, synchronize on both boundaries:```pythontorch.cuda.synchronize()start = time.perf_counter()output = model(batch)torch.cuda.synchronize()elapsed = time.perf_counter() - start```For low-overhead GPU timing, prefer CUDA events; they are timestamped on the GPU stream. Synchronize once before reading the result. Do **not** put `torch.cuda.synchronize()` after every operation in a real training loop: that destroys intended overlap and creates a serialized workload that the application never normally runs.#### CPU launch overhead versus slow kernelsMany tiny kernels can be slow even when no individual kernel is wide. Python and the dispatcher must prepare each launch, and the GPU can run out of queued work between launches. In a trace, suspect launch overhead when you see dense narrow kernels separated by gaps and a busy CPU launch thread. Test the hypothesis by increasing batch size or fusing operations with `torch.compile`; if gaps shrink and throughput rises, launch overhead was material. A single wide kernel points elsewhere.

In [ ]:
#  Part 1: torch.profiler
from torch.profiler import profile, record_function, ProfilerActivity

# Profile one forward+backward step
activities = [ProfilerActivity.CPU]

# Only ask the profiler to track CUDA kernels if a GPU is actually present
if HAS_GPU:
    activities.append(ProfilerActivity.CUDA)

optimizer.zero_grad()

# Wrap each phase in a named record_function span so the profiler table breaks costs down by phase
with profile(activities=activities, record_shapes=True, with_stack=False) as prof:
    with record_function("data_prep"):
        x_b = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
        y_b = torch.randint(0, VOCAB, (B, S)).to(DEVICE)

    with record_function("forward"):
        logits = model(x_b)
        loss = criterion(logits.view(-1, VOCAB), y_b.view(-1))

    with record_function("backward"):
        loss.backward()

    with record_function("optimizer_step"):
        optimizer.step()
        optimizer.zero_grad()

if HAS_GPU:
    torch.cuda.synchronize()

# Extract timings
key_averages = prof.key_averages()
print("Top 10 operators by CPU time:")
print(key_averages.table(sort_by="cpu_time_total", row_limit=10))


#### How to read this profiler table

| Column                | What it means                               | Action                                                 |
| --------------------- | ------------------------------------------- | ------------------------------------------------------ |
| `self_cpu_time_total` | Time _inside_ this op, excluding child ops  | **Sort by this** to find the actual hot op             |
| `cpu_time_total`      | Time including all child ops in the subtree | High here + low `self` = caller, not the bottleneck    |
| `cuda_time_total`     | Matching GPU kernel time                    | Large gap vs CPU = async overlap (usually good)        |
| `count`               | How many times this op ran                  | High count + low self = loop overhead, not kernel cost |

**Rule of thumb:** Sort by `self_cpu_time_total` to find what's actually slow. `cpu_time_total` shows where the call stack is deepest — useful for callers, not causes.


In [ ]:
#  Part 1b: Phase-level timing
phases = ["data_prep", "forward", "backward", "optimizer_step"]
phase_times = {}

# Pull each named phase's total CPU time out of the profiler's per-op event table
for phase in phases:
    events = [e for e in key_averages if e.key == phase]
    if events:
        t = events[0].cpu_time_total / 1000  # μs → ms
        phase_times[phase] = t
    else:
        phase_times[phase] = 0.0

total_t = sum(phase_times.values()) or 1.0

print("Phase timing breakdown:")
for phase, t in sorted(phase_times.items(), key=lambda x: -x[1]):
    pct = t / total_t * 100
    bar = "\u2588" * int(pct / 2)
    print(f"  {phase:20s}: {t:7.2f} ms  ({pct:5.1f}%)  {bar}")

# Determine actual order
sorted_phases = sorted(phase_times.items(), key=lambda x: -x[1])
order = [p for p, _ in sorted_phases]
print()
print(f"Actual order (slowest first): {' > '.join(order)}")
print()
print("Prediction check:")
print(
    "  Answer depends on hardware — the point is that the profiler told us, not our guess."
)
print(
    f"  {'backward' if order[0] == 'backward' else order[0]} is the dominant phase on this machine."
)

# Bar chart of the measured phase times, colored to match the earlier phase legend
fig, ax = plt.subplots(figsize=(9, 4))
color_map = {
    "data_prep": "steelblue",
    "forward": "#f0a500",
    "backward": "coral",
    "optimizer_step": "mediumseagreen",
}
color_list = [color_map.get(p, "gray") for p in phase_times.keys()]
bars = ax.bar(phase_times.keys(), phase_times.values(), color=color_list)
ax.set_ylabel("Time (ms)")
ax.set_title("Single step phase timing")

# Label each bar with its exact millisecond value
for bar, (p, t) in zip(bars, phase_times.items()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f"{t:.1f}ms",
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
plt.show()


#### What just happened — and what's missing

`torch.profiler` identified the most expensive phase and the top operators. The backward pass is typically 2–3× the forward pass time (it computes gradients for every parameter). On CPU, data movement to device shows up in data_prep; on GPU it's often negligible once data is already on device.

**Missing piece:** `torch.profiler` captures everything — including its own overhead. How much slower is profiled code vs. uninstrumented code? That's Part 2.


#### #### Your turn — how does batch size change the phase breakdown?

The phase timing above used `B=4, S=128`. Before changing anything, predict:

> If you **double the batch size** to `B=8`, which phase grows the most _proportionally_ — `forward`, `backward`, or `optimizer`?

Hint: all three scale linearly with `B` for most ops. But the AdamW optimizer step (momentum state update) has a component independent of batch size. What does that mean for its _percentage_ of total step time as you increase `B`?

**Then run the cell below** — change `B_YT` and see if your prediction holds.


In [ ]:
#  #### Your turn: phase timing at different batch/sequence lengths
B_YT = 8  # # CHANGE: try 4 (original), 8 (double batch), 16 (quadruple)
S_YT = 128  # # CHANGE: try 128 (original), 256 (doubles attention cost), 384

x_yt = torch.randint(0, VOCAB, (B_YT, S_YT)).to(DEVICE)
y_yt = torch.randint(0, VOCAB, (B_YT, S_YT)).to(DEVICE)
optimizer.zero_grad()

# Re-run the same phase-profiled step at the chosen batch/sequence length
with profile(activities=activities, record_shapes=False) as prof_yt:
    with record_function("forward_yt"):
        logits_yt = model(x_yt)
        loss_yt = criterion(logits_yt.view(-1, VOCAB), y_yt.view(-1))
    with record_function("backward_yt"):
        loss_yt.backward()
    with record_function("optimizer_yt"):
        optimizer.step()
        optimizer.zero_grad()

if HAS_GPU:
    torch.cuda.synchronize()
ka_yt = prof_yt.key_averages()
phase_yt = {p: 0.0 for p in ["forward_yt", "backward_yt", "optimizer_yt"]}

# Pull each named phase's CPU time out of this run's profiler events
for e in ka_yt:
    if e.key in phase_yt:
        phase_yt[e.key] = e.cpu_time_total / 1000

total_yt = sum(phase_yt.values()) or 1.0
print(f"Phase timing for B={B_YT}, S={S_YT}:")
for ph, t in sorted(phase_yt.items(), key=lambda x: -x[1]):
    pct = t / total_yt * 100
    print(f"  {ph:20s}: {t:7.2f} ms  ({pct:.1f}%)")

orig_bwd = phase_times.get("backward", 0)
new_bwd = phase_yt.get("backward_yt", 0)
ratio = new_bwd / orig_bwd if orig_bwd > 0 else float("nan")
print(f"\nOriginal backward (B={B}, S={S}): {orig_bwd:.2f} ms")
print(f"New backward (B={B_YT}, S={S_YT}): {new_bwd:.2f} ms  ({ratio:.1f}× change)")
print(
    f"→ Attention cost scales as S² — doubling S should roughly quadruple attention kernel time"
)
print(
    f"→ Forward+backward scale linearly with B — optimizer cost is mostly B-independent (parameter count)"
)


---

## Part 2 — `autograd.profiler`: Targeted Profiling

`torch.autograd.profiler.profile` is lighter-weight than `torch.profiler` — useful when you want to profile just one specific cell without the full trace overhead.

**Two-sided health check (Section 14.2):**

- **Too much profiling:** full `torch.profiler` on every training step slows the run by 5–15%
- **Too little:** no profiling at all means never knowing where the bottleneck is


#### #### Predict first — how much does the profiler slow things down?

When you wrap a forward+backward step in `torch.profiler`, the step gets slower — it intercepts every operator at the PyTorch dispatcher level. Before measuring, predict:

**By what factor does profiling slow down one complete training step?**

1. **(a) < 5%** — modern profilers are near-zero-cost; interception is O(1) per op
2. **(b) 5–25%** — noticeable but acceptable for occasional debugging runs
3. **(c) > 25%** — profiling fundamentally changes the behaviour you're measuring

If **(c)** is true, your profiler would make a fast phase appear slow — you'd be optimizing a mirage.


In [ ]:
#  Part 2: Profiling overhead measurement (two-sided health check)
def time_step(n_runs=20):
    """Time one forward+backward step without profiling."""
    if HAS_GPU:
        torch.cuda.synchronize()
    times = []

    # Repeat the uninstrumented step to get a stable median baseline
    for _ in range(n_runs):
        optimizer.zero_grad()
        if HAS_GPU:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model(x_batch)
        loss = criterion(out.view(-1, VOCAB), labels.view(-1))
        loss.backward()
        optimizer.step()
        if HAS_GPU:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000  # ms


def time_step_profiled(n_runs=5):
    """Time one forward+backward step with full profiler enabled."""
    times = []

    # Repeat the same step wrapped in torch.profiler to measure instrumentation cost
    for _ in range(n_runs):
        optimizer.zero_grad()
        t0 = time.perf_counter()
        with profile(activities=activities) as p:
            out = model(x_batch)
            loss = criterion(out.view(-1, VOCAB), labels.view(-1))
            loss.backward()
            optimizer.step()
        if HAS_GPU:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000


baseline_ms = time_step()
profiled_ms = time_step_profiled()
overhead_pct = (profiled_ms - baseline_ms) / baseline_ms * 100

print(f"Profiling overhead measurement:")
print(f"  Uninstrumented step:  {baseline_ms:.2f} ms")
print(f"  With torch.profiler:  {profiled_ms:.2f} ms")
print(f"  Overhead:             {overhead_pct:.1f}%")
print()

# Flag whether the measured overhead is acceptable for occasional profiling
if overhead_pct < 20:
    print(
        "\u2713 Health check (too-tight): profiling overhead is acceptable for occasional profiling runs"
    )
    print("  \u2192 Use torch.profiler on 1-2 steps, not every step")
else:
    print(
        f"! Overhead is {overhead_pct:.0f}% — only profile during debugging, not production training"
    )
print()
print("\u2713 Health check (too-loose): uninstrumented baseline gives reference timing")
print("  \u2192 Always run uninstrumented first to establish your baseline")


**Why does the profiler add overhead?**

The profiler intercepts every PyTorch operator call at the dispatcher level. Each interception records a timestamp, allocates a string (op name), and writes to the event buffer. The cost scales with **op-call count**, not compute time — so a model with many small ops (e.g., 1000 element-wise operations) sees more overhead than a model with few large matmuls (e.g., 5 large linear layers). This is why `with_stack=False` is faster than `with_stack=True`.


#### Profiler perturbation: measure the measuring toolInstrumentation adds event creation, timestamps, buffer writes, optional stack capture, and later trace serialization. The result is a **perturbed workload**, not a transparent recording. Compare an unprofiled baseline and a profiled run with the same warmup, inputs, synchronization boundaries, and repetition count.Concrete anti-patterns:- Profiling the first iteration and concluding that steady-state kernels are slow. The first iteration can include CUDA context creation, allocator growth, and compilation.- Enabling `with_stack=True`, shape recording, memory profiling, and a long active window simultaneously, then treating the resulting wall time as production latency.- Comparing a synchronized profiled run with an unsynchronized baseline.- Optimizing a 20 microsecond event whose duration changes by 15 microseconds between repeated captures.Use a short `wait -> warmup -> active` schedule, report the profiler slowdown beside every conclusion, and confirm improvements with the profiler disabled. A profile is excellent for forming a hypothesis; an uninstrumented benchmark is the acceptance test.

#### What just happened — and what's missing

You measured the observer effect: the profiler itself changed the behaviour it was observing. The key insight is that overhead scales with **operator-call count**, not with compute time — a matmul-heavy model with 5 large ops pays less overhead than an activation-heavy model with 500 element-wise ops. Using `with_stack=False` reduces it further.

**Missing piece:** You now know _how expensive_ each phase is (Part 1) and _how much the instrument distorts your measurement_ (Part 2). Both tell you _where_ time goes, not _why_ it goes there. Is the backward pass slow because of too many FLOPs, or because of too many memory roundtrips? Those two root causes require **opposite fixes** — and that's Part 3.


#### #### Your turn — observer effect on a simpler model

The profiling overhead depends on how many separate operator calls your model makes — a model with fewer ops pays more overhead _as a percentage_ of its own (already-short) run time. Before running the cell below, predict:

> Will a simple `Linear → GELU → Linear` model see **higher or lower** profiling overhead percentage compared to our 6-layer SmallTransformer?

Hint: SmallTransformer calls hundreds of operators per step; the simple model calls ~6. Each call has fixed bookkeeping cost regardless of its compute.


In [ ]:
#  #### Your turn: profiling overhead vs. operator density
simple_model = nn.Sequential(
    nn.Linear(D, D * 4), nn.GELU(), nn.Linear(D * 4, VOCAB)
).to(DEVICE)

# # CHANGE: try nn.Linear(D, VOCAB) alone for the extreme low-op case

x_flat = torch.randn(B * S, D).to(DEVICE)


# Time n forward passes, optionally wrapping each one in torch.profiler
def time_forward(m, inp, profiled=False, n=10):
    m.eval()
    times = []
    with torch.no_grad():
        for _ in range(n):
            if HAS_GPU:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            if profiled:
                with profile(activities=activities) as _p:
                    _ = m(inp)
            else:
                _ = m(inp)
            if HAS_GPU:
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return np.median(times) * 1000


t_simple_base = time_forward(simple_model, x_flat, profiled=False)
t_simple_prof = time_forward(simple_model, x_flat, profiled=True)
oh_simple = (t_simple_prof - t_simple_base) / t_simple_base * 100

print(f"Profiling overhead comparison:")
print(
    f"  Simple model   (3 ops):   base={t_simple_base:.2f} ms  profiled={t_simple_prof:.2f} ms  overhead={oh_simple:.1f}%"
)
print(
    f"  SmallTransformer (many):  base={baseline_ms:.2f} ms  profiled={profiled_ms:.2f} ms  overhead={overhead_pct:.1f}%"
)
print()

# Compare which model's overhead percentage is larger, and explain why
if oh_simple > overhead_pct:
    print(
        "→ Simple model has HIGHER % overhead: fewer ops = less useful work per profiler call"
    )
    print(
        "  The fixed bookkeeping cost per op shows up more clearly with sparse computation"
    )
else:
    print(
        "→ SmallTransformer has higher % overhead: more ops = more interception points"
    )
print(
    "→ Rule: profiler overhead = (fixed cost per op-call × op count) / total step time"
)


---## Part 3 — Compute-Bound vs. Memory-Bound: Which Limit Are We Hitting?Two operations look similar in source code but perform very differently:- **Matrix multiply** (Q @ K.T): many FLOPs per byte of memory → compute-bound- **Softmax** (along S axis): few FLOPs per byte → memory-boundA profiler showing high kernel time for softmax suggests a memory-bandwidth bottleneck (consistent with Ch1's roofline model: attention softmax has low arithmetic intensity).#### Before you inspect the roofline imageWhich point should move upward when memory bandwidth doubles but peak FLOP/s stays fixed: attention softmax, matmul, both, or neither? Commit to an answer, then use each point's position relative to the sloped and flat ceilings to check it.

![Roofline diagram: matmul (coral) sits near the compute ceiling; attention softmax (amber) sits far left in the memory-bound region](images/compute-vs-memory-bound.png)


### Walkthrough: classify the limit before choosing a fixIn the image, horizontal position is **arithmetic intensity**: useful FLOPs performed per byte moved from memory. The rising left side is the bandwidth ceiling; more reuse can move an operation right and raise performance. The flat top is the compute ceiling; once there, reducing memory traffic alone will not exceed the hardware's peak arithmetic rate.- **Attention softmax** sits left because it performs a few reductions and elementwise operations while repeatedly reading and writing the large score matrix. Likely experiment: fuse passes or avoid materializing the matrix, as FlashAttention does.- **Matmul** sits farther right because each loaded tile is reused for many multiply-accumulate operations. Likely experiment: use Tensor Core-friendly shapes and dtypes, then increase problem size enough to saturate the device.- A point below both ceilings is not automatically "memory-bound." It may reflect small shapes, launch overhead, dependencies, poor occupancy, or an inefficient kernel.```mermaidflowchart TD    A[Slow operator] --> B{GPU busy continuously?}    B -- No, gaps or tiny kernels --> C[Launch or input-pipeline bound]    B -- Yes --> D{Near bandwidth ceiling?}    D -- Yes --> E[Memory-bound]    D -- No --> F{Near compute ceiling?}    F -- Yes --> G[Compute-bound]    F -- No --> H[Check occupancy, shapes, dependencies, algorithm]    C --> I[Test fusion, batching, prefetch]    E --> J[Test fewer bytes or more reuse]    G --> K[Test lower precision or faster math]```**Do not classify from runtime alone.** A slow softmax is a clue, not proof of bandwidth saturation. Combine the trace with achieved bandwidth or FLOP/s from hardware counters, vary tensor shape, and change one resource demand at a time. If doubling arithmetic with similar bytes barely changes time, memory pressure is implicated; if runtime tracks FLOPs while bandwidth remains below peak, compute pressure is implicated.

#### #### Predict first — which attention operation is slower?

For one attention head with `(B=8, S=512, D=64)`, rank by wall-clock time:

1. **(a) `torch.matmul(Q, K.T)` is slower** — it performs O(S²·D) FLOPs, far more arithmetic than softmax
2. **(b) `torch.softmax(scores, dim=-1)` is slower** — despite fewer FLOPs, it reads and writes the full S×S matrix multiple times (numerics pass, normalization pass)
3. **(c) They take roughly equal time** — FLOPs and memory-bandwidth happen to balance at this configuration

By raw FLOP count, softmax does 5–10× fewer operations than the matmul. Does fewer FLOPs always mean faster?


In [ ]:
#  Part 3: Compare attention matmul vs. softmax timing
B_bench, S_bench, D_bench = 8, 512, 64
Q = torch.randn(B_bench, S_bench, D_bench).to(DEVICE)
K = torch.randn(B_bench, S_bench, D_bench).to(DEVICE)


# Generic op-timing helper: run op_fn() n times with optional GPU sync, return median ms + last result
def bench_op(op_fn, n=30, sync=HAS_GPU):
    if sync:
        torch.cuda.synchronize()
    times = []
    for _ in range(n):
        if sync:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = op_fn()
        if sync:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000, result


# Op 1: matmul Q @ K^T → high arithmetic intensity (compute-bound)
t_matmul, scores = bench_op(lambda: torch.matmul(Q, K.transpose(-2, -1)))

# Op 2: softmax over S dim → low arithmetic intensity (memory-bound)
t_softmax, _ = bench_op(lambda: torch.softmax(scores, dim=-1))

# Op 3: both together (standard attention)
t_full, _ = bench_op(
    lambda: torch.softmax(torch.matmul(Q, K.transpose(-2, -1)) / (D_bench**0.5), dim=-1)
)

matmul_flops = 2 * B_bench * S_bench * S_bench * D_bench / 1e9  # GFLOP
softmax_bytes = B_bench * S_bench * S_bench * 4 * 3 / 1e9  # 3x for read/compute/write

print(f"Attention operations at (B={B_bench}, S={S_bench}, D={D_bench}):")
print(
    f"  Q @ K^T (matmul):  {t_matmul:.3f} ms  |  {matmul_flops/t_matmul*1000:.1f} GFLOPS  (likely compute-bound)"
)
print(
    f"  softmax(scores):   {t_softmax:.3f} ms  |  {softmax_bytes/t_softmax*1000:.1f} GB/s effective BW"
)
print(f"  Full attention:    {t_full:.3f} ms")
print()

# Reference peaks for interpretation
print("\n→ Hardware reference peaks (for context):")
print("   A100 SXM: ~312 TFLOPS (fp16) peak compute, 2.0 TB/s peak bandwidth")
print("   RTX 4090: ~330 TFLOPS (fp16) peak compute, 1.0 TB/s peak bandwidth")
print("   A10G:     ~125 TFLOPS (fp16) peak compute, 0.6 TB/s peak bandwidth")
print(f"\n→ Your measured GFLOPS / 312,000 = fraction of A100 compute used")
print("→ If < 10%: memory-bound (fix memory access pattern)")
print("→ If > 50%: compute-bound (batch larger, use tensor cores)")

# Whichever op measured slower reveals whether this run is compute- or memory-bound
if t_matmul < t_softmax:
    print(
        "\u2192 On this machine: softmax takes LONGER than the matmul despite fewer FLOPs"
    )
    print(
        "  This is the memory-bandwidth bottleneck: softmax reads/writes a (S\u00d7S) matrix repeatedly"
    )
    print(
        "  FlashAttention (Ch4) solves exactly this: keeps the S\u00d7S intermediate in SRAM"
    )
else:
    print("\u2192 On this machine: matmul dominates (likely a small GPU or CPU)")
    print(
        "  On data-center GPUs (A100, H100): softmax often becomes the bottleneck at S=2048+"
    )


#### What just happened — and what's missing

The profiler revealed that softmax over the attention matrix can be slower than the matmul on memory-bandwidth-limited hardware. The S×S matrix must be read and written multiple times (numerics stabilization, normalization) — each pass is a full HBM roundtrip.

**Missing piece:** We identified the bottleneck, but haven't fixed it. Can we fuse the matmul and softmax so the S×S matrix never leaves SRAM? Yes — that's FlashAttention (Ch4). But first, how much does `torch.compile` help without changing the algorithm?


#### #### Your turn — find the softmax crossover sequence length

At short sequences (S=64), matmul likely dominates. At long sequences (S=2048), softmax may dominate because the S×S intermediate matrix saturates memory bandwidth. Find the crossover point.

**Predict:** at what sequence length does softmax time first exceed matmul time on your hardware?

Then change `seq_lengths` in the cell below to test your prediction.


In [ ]:
#  #### Your turn: matmul vs. softmax crossover at different sequence lengths
seq_lengths = [64, 128, 256, 512, 1024]  # # CHANGE: add 2048 if memory allows

print(f"  {'S':>5}  {'matmul (ms)':>12}  {'softmax (ms)':>13}  {'softmax/matmul':>14}")
print("  " + "-" * 50)
crossover_found = False

# Sweep S and find the point where softmax overtakes matmul as the slower op
for S_t in seq_lengths:
    Q_t = torch.randn(4, S_t, 64).to(DEVICE)
    K_t = torch.randn(4, S_t, 64).to(DEVICE)
    t_mm, sc_t = bench_op(lambda: torch.matmul(Q_t, K_t.transpose(-2, -1)), n=20)
    t_sm, _ = bench_op(lambda: torch.softmax(sc_t, dim=-1), n=20)
    ratio = t_sm / t_mm if t_mm > 0 else 0
    marker = " ← CROSSOVER" if (t_sm > t_mm and not crossover_found) else ""
    if t_sm > t_mm:
        crossover_found = True
    print(f"  S={S_t:>4}:  {t_mm:>10.3f} ms  {t_sm:>11.3f} ms  {ratio:>12.2f}×{marker}")

print()
if crossover_found:
    print(
        "→ Crossover found: softmax becomes the memory-bandwidth bottleneck past that S"
    )
else:
    print("→ No crossover at these sequence lengths on this hardware")
    print("  On A100/H100 with S ≥ 2048, softmax typically becomes the bottleneck")
print(
    "→ FlashAttention solves the crossover: keeps the S×S scores in SRAM (no HBM roundtrip)"
)


---

## Part 4 — `torch.compile`: Graph Compilation

`torch.compile` (PyTorch 2.0) compiles a model using TorchDynamo (graph capture) and inductor (backend optimization). It fuses operators, removes Python overhead, and generates optimized kernels — without changing your code.

**When it helps:** Models with many small ops (lots of element-wise operations, activations).
**When it doesn't:** Models that are already memory-bandwidth-limited; the bottleneck is hardware, not Python overhead.


#### #### Predict first — how much does `torch.compile` speed up inference?

For our 6-layer SmallTransformer (~5M params), `torch.compile(mode='default')` will produce:

1. **(a) 0–5% speedup** — Python overhead is negligible; the model already spends all its time in C++/CUDA kernels
2. **(b) 10–30% speedup** — meaningful: op fusion and Triton kernel generation remove Python dispatch overhead
3. **(c) > 2× speedup** — compile transforms the runtime fundamentally at this model size

Also predict: **how many forward passes does it take to break even on the 30–90s compilation cost?**


In [ ]:
#  Part 4: torch.compile speedup measurement
model_eager = SmallTransformer().to(DEVICE)
model_eager.load_state_dict(model.state_dict())  # same weights

# torch.compile may be unavailable on older PyTorch versions or unsupported platforms
try:
    model_compiled = torch.compile(SmallTransformer().to(DEVICE), mode="default")
    model_compiled.load_state_dict(model.state_dict())
    COMPILE_AVAILABLE = True
    print("torch.compile available")
except Exception as e:
    print(f"torch.compile not available: {e}")
    COMPILE_AVAILABLE = False


# Time inference-only forward passes after a warm-up period
def time_inference(m, n_runs=30, warmup=5):
    m.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = m(x_batch)
        if HAS_GPU:
            torch.cuda.synchronize()
        times = []
        for _ in range(n_runs):
            if HAS_GPU:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = m(x_batch)
            if HAS_GPU:
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return np.median(times) * 1000


t_eager = time_inference(model_eager)
print(f"\nEager mode (inference): {t_eager:.2f} ms")

# Only benchmark the compiled model if torch.compile actually succeeded
if COMPILE_AVAILABLE:

    # First call triggers compilation (warm-up cost)
    print("Compiling model (first call triggers compilation)...")
    t0_compile = time.perf_counter()
    model_compiled.eval()
    with torch.no_grad():
        _ = model_compiled(x_batch)
    compile_overhead_s = time.perf_counter() - t0_compile

    t_compiled = time_inference(model_compiled)
    speedup = t_eager / t_compiled

    print(
        f"Compiled mode (inference): {t_compiled:.2f} ms  ({speedup:.1f}\u00d7 speedup after warm-up)"
    )
    print(f"Compilation overhead: {compile_overhead_s:.1f}s (one-time cost)")
    print()
    if speedup > 1.1:
        print("\u2192 torch.compile improved throughput on this architecture")
    else:
        print(
            "\u2192 Speedup is modest \u2014 likely memory-bandwidth-limited (compile helps compute, not bandwidth)"
        )
    print(
        f"  Break-even at {compile_overhead_s / max(t_eager - t_compiled, 0.001) * 1000:.0f} forward passes"
    )
else:
    print(
        "\n\u2192 Use torch.compile when: many small ops, compute-bound, model runs >1000 times"
    )
    print(
        "  Skip when: memory-bandwidth-limited, model architecture changes frequently"
    )


#### What just happened — and what's missing

`torch.compile` traded a one-time compilation cost for lower per-step latency. Your break-even calculation shows whether it's a viable trade: running 100,000 training steps with a 15% speedup breaks even in ~1,000 steps — almost always worth it for a production run.

**Missing piece:** Compile helps by fusing operators and generating optimised Triton kernels — it removes Python dispatch overhead. But if the bottleneck is hardware _memory bandwidth_ (as Part 3 showed for softmax at large S), no amount of fusion helps: the S×S matrix still needs to cross the memory bus. The next question is: how do you isolate _exactly which sub-operation within a phase_ is bleeding time? That requires surgical profiling with named regions — Part 5.


#### #### Your turn — compare `torch.compile` modes

`torch.compile` supports three modes with different speed/compile-time tradeoffs:

- `'default'` — safe, fast compilation (~30s)
- `'reduce-overhead'` — minimises Python dispatch overhead, especially effective for models with many small ops (~45s to compile)
- `'max-autotune'` — exhaustive Triton kernel search, slowest to compile but fastest inference (~5–10 min)

**Predict:** which mode gives the best inference speedup for our transformer? Which mode makes the _biggest relative improvement_ for a simple model vs. a transformer?

Add `'max-autotune'` to the `modes_to_test` list below when you have time to wait for compilation.


In [ ]:
#  #### Your turn: torch.compile mode comparison
# Only run the mode comparison if torch.compile is actually available on this machine
if COMPILE_AVAILABLE:

    # # CHANGE: add 'max-autotune' to modes list (takes 5–10 min to compile)
    modes_to_test = ["default", "reduce-overhead"]

    print("torch.compile mode comparison (inference only):")
    print(
        f"  {'Mode':>18}  {'Compile (s)':>12}  {'Inference (ms)':>15}  {'vs eager':>10}"
    )
    print("  " + "-" * 62)

    # Compile and time each candidate mode, skipping any that fail on this hardware
    for mode_name in modes_to_test:
        try:
            m_c = torch.compile(SmallTransformer().to(DEVICE), mode=mode_name)
            m_c.load_state_dict(model.state_dict())
            t_start = time.perf_counter()
            with torch.no_grad():
                _ = m_c(x_batch)  # trigger compilation
            compile_s = time.perf_counter() - t_start
            t_c = time_inference(m_c)
            sp = t_eager / t_c
            print(
                f"  {mode_name:>18}:  {compile_s:>10.1f}s  {t_c:>13.2f}ms  {sp:>8.2f}×"
            )
        except Exception as e:
            print(f"  {mode_name:>18}: unavailable — {e}")

    print(f"  {'eager (baseline)':>18}:  {'—':>10}   {t_eager:>13.2f}ms  {'1.00':>8}×")
    print()
    print("→ 'reduce-overhead' excels for models with many small Python-dispatched ops")
    print(
        "→ 'max-autotune' exhaustively searches Triton kernel configs — worth it for production serving (millions of steps)"
    )
else:
    print("torch.compile not available — requires PyTorch ≥ 2.0")
    print(
        "→ Expected: max-autotune > reduce-overhead > default > eager for compute-bound models"
    )


---

## Part 5 — Custom Profiling Regions with `record_function`

`torch.profiler` captures _everything_ — setup code, logging calls, evaluation steps you don't care about. When Part 1 already told you the backward pass owns 70% of wall time, you don't need a full operator timeline; you need **surgical visibility** into just that one region.

`record_function("name")` inserts a named span that appears in the Chrome trace without the overhead of tracing the full operator tree. Use it to answer: "Within the backward pass, which _specific_ layer's gradient is the true bottleneck?" — the difference between a full blood panel (`torch.profiler`) and a targeted biopsy (`record_function`).

This is exactly what production ML engineers do: run `torch.profiler` once to find the slow _phase_ (Part 1), then drop `record_function` inside that phase to find the slow _operator_ (Part 5).


#### #### Predict first — which training sub-step is slowest?

We are about to time six sub-steps: `tokenize → to_device → forward → loss → backward → optimizer_step`.

On **CPU hardware** (no PCIe GPU transfer cost), predict the rank order from slowest to fastest:

1. **(a) backward > forward > optimizer > to_device > loss > tokenize** — gradient computation dominates; data is already in RAM so device transfer is cheap
2. **(b) forward > backward > to_device > optimizer > loss > tokenize** — inference pass is expensive with a 6-layer model
3. **(c) to_device > backward > forward > optimizer > loss > tokenize** — even CPU tensor copies dominate

Hint: `to_device` on CPU is a near-zero-cost view (data stays in RAM). How does that constrain the ranking?


In [ ]:
#  Part 5: Custom profiling regions
# Simulate a realistic training step with multiple sub-operations
N_STEPS_PROFILE = 3
step_times = {
    "tokenize": [],
    "to_device": [],
    "forward": [],
    "loss": [],
    "backward": [],
    "step": [],
}

# Nest a named record_function span around every sub-operation of each simulated step
with profile(activities=activities, record_shapes=False) as prof_detailed:
    for step in range(N_STEPS_PROFILE):

        with record_function(f"step_{step}/tokenize"):
            t0 = time.perf_counter()

            # Simulate tokenization (CPU-bound)
            x_cpu = torch.randint(0, VOCAB, (B, S))
            step_times["tokenize"].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/to_device"):
            t0 = time.perf_counter()
            x_gpu = x_cpu.to(DEVICE)
            y_gpu = torch.randint(0, VOCAB, (B, S)).to(DEVICE)
            if HAS_GPU:
                torch.cuda.synchronize()
            step_times["to_device"].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/forward"):
            t0 = time.perf_counter()
            logits = model(x_gpu)
            if HAS_GPU:
                torch.cuda.synchronize()
            step_times["forward"].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/loss"):
            t0 = time.perf_counter()
            loss_val = criterion(logits.view(-1, VOCAB), y_gpu.view(-1))
            step_times["loss"].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/backward"):
            t0 = time.perf_counter()
            optimizer.zero_grad()
            loss_val.backward()
            if HAS_GPU:
                torch.cuda.synchronize()
            step_times["backward"].append((time.perf_counter() - t0) * 1000)

        with record_function(f"step_{step}/optimizer_step"):
            t0 = time.perf_counter()
            optimizer.step()
            if HAS_GPU:
                torch.cuda.synchronize()
            step_times["step"].append((time.perf_counter() - t0) * 1000)

print("Detailed step timing (median across 3 steps):")
total_recorded = 0
for op, times_list in step_times.items():
    t = np.median(times_list)
    total_recorded += t
    print(f"  {op:20s}: {t:.3f} ms")
print(f"  {'TOTAL':20s}: {total_recorded:.3f} ms")
print()
print(
    "\u2192 record_function regions appear as named spans in the Chrome trace viewer."
)
print("  Open the saved .json trace at: chrome://tracing  (or ui.perfetto.dev)")

# Save trace
trace_path = "profiler_trace.json"
prof_detailed.export_chrome_trace(trace_path)
print(f"  Trace saved to: {os.path.abspath(trace_path)}")


#### Reading your Chrome trace in 5 steps1. Open `chrome://tracing` in Chrome (or [ui.perfetto.dev](https://ui.perfetto.dev) for a modern UI)2. Click **Load** (top-left button) → select the `profiler_trace.json` file printed above3. **Rows** = CPU threads; GPU kernels appear as additional rows labeled with CUDA stream names4. Your `record_function` names appear as **colored spans** — look for `step_N/backward`, `step_N/forward`, etc.5. Navigate: **W/S** = zoom in/out, **A/D** = pan left/right; click any span for exact duration in the bottom panel**What to look for first:** The widest span in the backward row — that's your current bottleneck. A gap means no recorded GPU work occupied that lane during the interval. Correlate it with CPU lanes, dependencies, synchronization, copies, and other streams before blaming launch delay.

#### From a trace symptom to a defensible conclusionA trace is a time-ordered record of observed events. Start with a visible symptom, choose a measurement that could distinguish causes, state a falsifiable hypothesis, and change one variable.```mermaidflowchart LR    S[Symptom: low throughput] --> M[Measurement: short CPU and GPU trace]    M --> H[Hypothesis: CPU cannot launch fast enough]    H --> E[Experiment: fuse ops or increase batch size]    E --> R{Gaps shrink and throughput rises?}    R -- Yes --> C[Hypothesis supported]    R -- No --> N[Revise hypothesis and measure again]    C --> V[Validate without profiler]```**Concrete trace patterns:**| Pattern | Plausible hypothesis | Discriminating experiment || --- | --- | --- || GPU lane has periodic blank regions while a DataLoader thread is busy | Input pipeline starvation | Increase workers or prefetch; check whether blank regions shrink || CPU launches many tiny kernels and the GPU lane has micro-gaps | CPU launch overhead | Fuse operations or increase batch size; compare throughput and gap density || One kernel is wide and the GPU stays busy | Kernel implementation or resource limit | Add hardware counters; vary dtype and shape || Copy and compute appear on different streams but never overlap | Missing pinned memory, dependency, or synchronization | Use pinned input memory and `non_blocking=True`; inspect dependencies |#### What traces cannot prove- A wide span does not prove a kernel is inefficient; it may simply perform the most necessary work.- A gap does not prove the CPU is at fault; a dependency, explicit synchronization, allocator event, communication, or missing instrumentation can also create it.- Operator names and durations do not prove compute-bound versus memory-bound behavior without hardware counters or controlled scaling experiments.- One capture does not establish a stable regression or causal relationship. Scheduling noise, thermal state, clocks, input shape, compilation, and profiler overhead can change the picture.- Absence of an event does not prove absence of work; the selected activities, threads, streams, or external libraries may not be instrumented.Anti-pattern: zoom into the widest colored block and immediately rewrite it. Better: inspect parent and child spans, CPU-to-GPU launch arrows, stream dependencies, repeated-step consistency, and the unprofiled benchmark before accepting a cause.

#### What just happened — and what's missing`record_function` gave you named spans at the training-sub-step level — exactly the granularity needed to close in on the bottleneck. The Chrome trace shows each span as a coloured block; **gaps** reveal intervals with no recorded work on that GPU lane. CPU dispatch delay, data starvation, dependencies, synchronization, communication, or missing instrumentation can all produce that symptom. Correlate lanes first; then test targeted fixes such as prefetching or `non_blocking=True` transfers.**Missing piece:** Everything so far was profiled on a _single isolated step_. A real training loop has variance: the first 5–10 steps are always slower due to CUDA context init, JIT warmup, and memory caching. Reliable bottleneck measurement requires timing across many DataLoader iterations and discarding warmup — that's Part 6.

#### #### Your turn — find the `to_device` bottleneck on GPU

On GPU, moving tensors from CPU RAM to GPU VRAM via the PCIe bus is not free. The `pin_memory=True` DataLoader option pre-allocates CPU memory in a form that the GPU can DMA-copy directly without OS involvement.

**Predict:** for our in-memory tensor dataset (already in RAM, no disk I/O), will `pin_memory=True` reduce `to_device` time measurably?


In [ ]:
#  #### Your turn: pin_memory effect on to_device time
# # CHANGE: flip pin_memory_on between True and False to compare


# Time repeated batch transfers to DEVICE, with pin_memory on the DataLoader toggled on/off
def measure_to_device(pin_memory_on, n=20):
    dl = DataLoader(
        dataset,
        batch_size=B,
        shuffle=False,
        num_workers=0,
        pin_memory=(pin_memory_on and HAS_GPU),
    )
    it = iter(dl)
    times = []
    for _ in range(n):
        xb, _ = next(it)
        if HAS_GPU:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = xb.to(DEVICE, non_blocking=True)
        if HAS_GPU:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000


t_no_pin = measure_to_device(pin_memory_on=False)
t_pinned = measure_to_device(pin_memory_on=True)
speedup_pin = t_no_pin / t_pinned if t_pinned > 0 else 1.0

print(f"to_device timing:")
print(f"  Without pin_memory: {t_no_pin:.3f} ms")
print(f"  With    pin_memory: {t_pinned:.3f} ms  ({speedup_pin:.2f}× speedup)")
print()

# Interpretation differs depending on whether a real GPU transfer is happening
if not HAS_GPU:
    print(
        "→ No GPU: to_device is a near-zero-cost view copy on CPU (data already in RAM)"
    )
    print(
        "  Real benefit of pin_memory: only on GPU with data loaded from disk (SSD → DRAM → GPU VRAM)"
    )
elif speedup_pin > 1.1:
    print(f"→ pin_memory gave {speedup_pin:.2f}× speedup on GPU transfer")
    print("  Enable pin_memory whenever your dataset fits in RAM")
else:
    print(
        "→ Minimal difference for in-memory tensor data (PCIe pipeline is not the bottleneck here)"
    )
    print(
        "  Larger benefit on disk-backed datasets (image files, HDF5) with num_workers > 0"
    )


---

## Part 6 — End-to-End: Profiling a Real Training Step

Now we profile a step closer to what the `02-llm-finetuning` fine-tuning loop does: load a batch from a DataLoader, run forward/backward, and time each phase.

#### #### Predict first

Across a full training step (data loading + forward + backward + optimizer), which component typically surprises practitioners most?

1. **(a) Data loading** — they assume the GPU is always busy; disk/CPU preprocessing can be the bottleneck
2. **(b) The optimizer step** — AdamW with momentum state updates is often neglected
3. **(c) The forward pass** — practitioners often focus on backward, underestimating forward complexity

The answer varies by hardware and batch size — the profiler tells you which is true for YOUR setup.


In [ ]:
#  Part 6: End-to-end step timing with DataLoader
from torch.utils.data import TensorDataset, DataLoader

# Simulate a realistic dataset (in-memory for this demo)
N_SAMPLES = 1000
dummy_x = torch.randint(0, VOCAB, (N_SAMPLES, S))
dummy_y = torch.randint(0, VOCAB, (N_SAMPLES, S))
dataset = TensorDataset(dummy_x, dummy_y)
loader = DataLoader(
    dataset, batch_size=B, shuffle=True, num_workers=0, pin_memory=HAS_GPU
)

phase_records = {
    k: [] for k in ["load_batch", "to_device", "forward", "backward", "optimizer"]
}

model.train()
loader_iter = iter(loader)
N_MEASURED = min(20, len(loader))

# Time every phase of a real training step, batch-loading included, for N_MEASURED steps
for step in range(N_MEASURED):
    t0 = time.perf_counter()
    x_cpu, y_cpu = next(loader_iter)
    if HAS_GPU:
        torch.cuda.synchronize()
    phase_records["load_batch"].append((time.perf_counter() - t0) * 1000)

    t1 = time.perf_counter()
    x_dev, y_dev = x_cpu.to(DEVICE), y_cpu.to(DEVICE)
    if HAS_GPU:
        torch.cuda.synchronize()
    phase_records["to_device"].append((time.perf_counter() - t1) * 1000)

    t2 = time.perf_counter()
    logits = model(x_dev)
    loss_val = criterion(logits.view(-1, VOCAB), y_dev.view(-1))
    if HAS_GPU:
        torch.cuda.synchronize()
    phase_records["forward"].append((time.perf_counter() - t2) * 1000)

    t3 = time.perf_counter()
    optimizer.zero_grad()
    loss_val.backward()
    if HAS_GPU:
        torch.cuda.synchronize()
    phase_records["backward"].append((time.perf_counter() - t3) * 1000)

    t4 = time.perf_counter()
    optimizer.step()
    if HAS_GPU:
        torch.cuda.synchronize()
    phase_records["optimizer"].append((time.perf_counter() - t4) * 1000)

# Summary
medians = {k: np.median(v) for k, v in phase_records.items()}
total_ms = sum(medians.values())

print(f"End-to-end training step breakdown (median over {N_MEASURED} steps):")
print()
print(f"{'Phase':20s}  {'Time (ms)':10s}  {'% of total':10s}  {'Cumulative':10s}")
print("-" * 55)
cumulative = 0

# Identify the single slowest phase to target for optimization
bottleneck = max(medians, key=medians.get)
for phase, ms in sorted(medians.items(), key=lambda x: -x[1]):
    pct = ms / total_ms * 100
    cumulative += pct
    flag = " \u2190 BOTTLENECK" if phase == bottleneck else ""
    print(f"  {phase:18s}  {ms:8.2f}ms  {pct:8.1f}%  {cumulative:8.1f}%{flag}")
print(f"\n  Total step time: {total_ms:.2f} ms  ({1000/total_ms:.1f} steps/second)")

# The bottleneck and recommended fix
print()
print(
    f"Closing Decision: The bottleneck is '{bottleneck}' at {medians[bottleneck]/total_ms*100:.0f}% of wall time."
)

# Map the measured bottleneck phase to a concrete, chapter-linked optimization
if bottleneck == "backward":
    recommendation = "Use gradient checkpointing (Ch2) or mixed precision (Ch2)"
    saving = 25
elif bottleneck == "load_batch":
    recommendation = "Use num_workers > 0 in DataLoader to prefetch on CPU"
    saving = 40
elif bottleneck == "forward":
    recommendation = "Try torch.compile or FlashAttention (Ch4)"
    saving = 20
else:
    recommendation = "Profile with more steps to confirm; consider mixed precision"
    saving = 15
print(f"Recommendation: {recommendation}")
print(f"Expected saving: ~{saving}%")

print()
print(f"--- Closing the 45-second mystery ---")
steps_per_epoch = 1000  # ← substitute len(train_loader) for your actual loader
epoch_s_estimate = (total_ms / 1000) * steps_per_epoch
print(f"  Assuming {steps_per_epoch} steps/epoch:")
print(f"  Estimated epoch time: ~{epoch_s_estimate:.0f}s")
print(
    f"  The '{bottleneck}' phase owns ~{(medians[bottleneck]/1000 * steps_per_epoch):.0f}s of that."
)
print(f"  (Substitute: steps_per_epoch = len(your_train_loader))")


#### What just happened — and what's missing

You measured the _average_ step cost across 20 real DataLoader iterations — this is the number that matters for epoch-time estimates. The bottleneck you found is the profiler's definitive answer for your hardware configuration. On CPU, backward typically owns 40–60% of step time; on data-centre GPUs, forward and backward compress towards parity while `to_device` can vanish (DMA saturates PCIe and overlaps with GPU compute).

**Missing piece:** We measured the average step. But step 0 is always anomalously slow: CUDA context initialization, first-batch JIT compilation, and cold caches all inflate it. Production profiling always discards warmup steps — `torch.profiler`'s `skip_first` + `wait`/`warmup`/`active` schedule exists precisely for this. Always discard at least 5 warmup steps before recording the numbers you report.


#### #### Your turn — does `num_workers` help for in-memory data?

The `num_workers=0` DataLoader in Part 6 fetches batches on the main thread. With `num_workers > 0`, Python spawns worker processes that prefetch batches while the GPU is busy — hiding the data-loading latency.

**Predict:** for our **in-memory tensor dataset** (no disk I/O), will `num_workers=2` reduce `load_batch` time? What about for a real dataset loaded from disk (images, HDF5)?


In [ ]:
#  #### Your turn: DataLoader num_workers comparison
# # CHANGE: add 4 to the list to test — add more workers = more prefetch parallelism
num_workers_configs = [0, 2]

print("DataLoader num_workers comparison (in-memory dataset):")
print(f"  {'num_workers':>12}  {'load+transfer (ms)':>20}  {'total step (ms)':>17}")
print("  " + "-" * 55)

# Compare load+transfer and full step time across different worker-process counts
for nw in num_workers_configs:
    try:
        dl_nw = DataLoader(
            dataset,
            batch_size=B,
            shuffle=False,
            num_workers=nw,
            pin_memory=(nw > 0 and HAS_GPU),
        )
        load_t, step_t = [], []
        it = iter(dl_nw)
        for _ in range(min(15, len(dl_nw))):
            t0 = time.perf_counter()
            xb, yb = next(it)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if HAS_GPU:
                torch.cuda.synchronize()
            t_load_val = (time.perf_counter() - t0) * 1000
            load_t.append(t_load_val)

            t1 = time.perf_counter()
            optimizer.zero_grad()
            out_nw = model(xb)
            loss_nw = criterion(out_nw.view(-1, VOCAB), yb.view(-1))
            loss_nw.backward()
            optimizer.step()
            if HAS_GPU:
                torch.cuda.synchronize()
            step_t.append(t_load_val + (time.perf_counter() - t1) * 1000)

        print(
            f"  {nw:>12}:  {np.median(load_t):>18.2f}ms  {np.median(step_t):>15.2f}ms"
        )
    except Exception as e:
        print(f"  {nw:>12}: error — {e}")

print()
print(
    "→ In-memory dataset: num_workers spawn overhead may INCREASE total time for small batches"
)
print(
    "→ Real benefit: image or text datasets decoded from disk — workers load next batch while GPU trains current one"
)
print("→ Rule of thumb: set num_workers = (CPU cores) / 2 for disk-backed datasets")


---

## Toy → Production Bridge

Everything in this notebook ran on a ~5M-parameter transformer on a single CPU (or consumer GPU). Production ML looks very different — but the profiling _workflow_ is identical.

| Toy context (this notebook)              | Production equivalent                               | Why the pattern transfers                                                         |
| ---------------------------------------- | --------------------------------------------------- | --------------------------------------------------------------------------------- |
| `B=4, S=128, D=256`, 6 layers            | `B=64–256, S=2048, D=4096` (GPT-3 scale)            | backward ≈ 2–3× forward holds at every scale                                      |
| CPU `torch.profiler`                     | CUDA + NCCL profiler on A100/H100 cluster           | Same `record_function` API; GPU adds `cuda_time_total` column                     |
| Softmax memory-bound at S=512            | FlashAttention needed at S=2048–128k                | Memory wall worsens as S²; crossover moves earlier on faster GPUs                 |
| `torch.compile mode='default'`           | `mode='max-autotune'` + Triton kernel search        | Autotune searches kernel configs; 10-min compile amortised over millions of steps |
| `DataLoader num_workers=0`               | `num_workers=8, prefetch_factor=4`                  | Multi-GPU needs a prefetch pipeline to hide PCIe latency at 400 GB/s              |
| `profiler_trace.json` → chrome://tracing | Nsight Systems `.qdrep` trace                       | Nsight adds SM utilization, warp stalls, memory bandwidth per kernel              |
| Single-node, single-GPU step timing      | DDP AllReduce timing (8–1000 GPUs)                  | AllReduce becomes its own bottleneck phase at ≥ 8 GPUs                            |
| `time.perf_counter` manual timer         | `wandb.log({"step_ms": ...})` continuous monitoring | Surfaces regressions across thousands of training runs automatically              |

**The rule that scales:** `measure first, guess last`. The same `torch.profiler` workflow used here is what ML engineers use on 1,000-GPU training runs. The only difference is the `cuda_time_total` column gets more interesting.


---

## Summary and Closing Decision

| Part | Tool | Key finding |
|------|------|------------|
| 1 | `torch.profiler` | Backward is 2–3× the cost of forward — the ratio is structural (gradient graph traversal), not a coincidence |
| 2 | Profiling overhead | Overhead scales with op-call *count*, not compute time — small-op-heavy models pay more overhead per unit of useful work |
| 3 | Compute vs. memory bound | Softmax becomes slower than matmul as S grows: S² memory accesses saturate bandwidth before FLOPs saturate compute |
| 4 | `torch.compile` | Break-even at ~600 passes for 15% speedup — worth it for production; negligible for memory-bound kernels |
| 5 | `record_function` | Surgical spans add < 1% overhead; **gaps** between Chrome trace spans reveal idle GPU time |
| 6 | End-to-end timing | DataLoader stalls can own 30–50% of wall time on disk-backed datasets — invisible to operator-level profilers |

### Key insights to keep

- **"Measure first, guess last"** — a single `torch.profiler` run reveals the real bottleneck faster than a week of trial-and-error optimization.
- **"The observer distorts the measurement"** — profiling overhead scales with operator count; a small-op-heavy model pays more overhead per useful computation than a matmul-heavy one.
- **"Memory-bound and compute-bound require opposite fixes"** — applying `torch.compile` to a bandwidth-limited softmax wastes compile time and may worsen performance slightly.
- **"Backward ≈ 2–3× forward — always"** — gradient computation traverses the full computation graph in reverse; this ratio is structural, not a bug.
- **"Chrome trace gaps are money left on the table"** — idle GPU time between `record_function` spans means the CPU is blocking GPU dispatch; fix with async data loading and `non_blocking=True` transfers.
- **"Break-even math matters"** — `torch.compile`'s ~60s compilation cost only pays off after hundreds of inference steps; calculate before you commit.

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- `torch.profiler` — captured CPU+CUDA operator timeline; extracted phase breakdowns
- Profiling overhead — measured two-sided: too much profiling slows training; none = flying blind
- Compute vs. memory bound — softmax vs. matmul compared; memory bottleneck identified
- `torch.compile` — speedup measured; break-even calculated
- `record_function` — custom regions created; Chrome trace saved
- End-to-end step timing — per-phase percentages measured on a real DataLoader

### Tier 2 — Explained but Not Built

- **PyTorch Memory Snapshot** — `torch.cuda.memory._snapshot()` captures full allocation graph; shown but not built

### Tier 3 — Named but Out of Scope

- **Nsight Systems** — NVIDIA's production GPU profiler; shows CUDA kernels, PCIe transfers, and SM utilization at hardware level
- **Perfetto** — Google's open-source trace viewer; compatible with Chrome trace format
- **DCGM** — Data Center GPU Manager; cluster-wide GPU utilization monitoring


---

## When to Use What

| Situation                                  | Tool                                    | Why                                     |
| ------------------------------------------ | --------------------------------------- | --------------------------------------- |
| "Something is slow, I don't know what"     | `torch.profiler` for 3–5 steps          | Full operator timeline                  |
| "I want to time one function quickly"      | `record_function` + `time.perf_counter` | Low overhead                            |
| "Is my GPU actually busy?"                 | `torch.cuda.utilization()`              | Hardware utilization check              |
| "My model uses lots of small ops"          | `torch.compile`                         | Fuses ops; removes Python overhead      |
| "Why is attention slow at long sequences?" | FlashAttention (Ch4)                    | Algorithmic fix: no S×S materialization |

→ **Next:** `learning/ai-infrastructure/04-flash-attention/` — profiling showed that attention softmax is memory-bound at S≥512. The next chapter explains exactly _why_ and how FlashAttention solves it without any accuracy change.


---

## Production and Cloud Profiling Practice

A useful production profile starts with a **representative workload contract**: pin the model and code revision, precision, device type, batch and sequence-shape distribution, data path, concurrency, and compilation settings. Reproduce the serving or training mix instead of optimizing a single convenient tensor shape. Warm the runtime before measuring so CUDA context creation, allocator growth, kernel compilation, cache fill, and data-loader startup do not become the baseline.

Full traces are diagnostic artifacts, not always-on telemetry. Capture a short scheduled window from a small, randomized share of healthy instances or from a canary job; keep cheap counters, latency histograms, throughput, memory, utilization, queue depth, and error rates continuously. Bound profiler overhead with `wait`/`warmup`/`active` schedules, a small repeat count, disabled stacks and shape recording by default, explicit artifact-size limits, and one capture at a time. Measure the profiler's own slowdown before increasing sample frequency.

Treat performance like correctness:

- Store a versioned baseline for representative cases and define regression budgets for median or p95 step latency, throughput, peak memory, and selected operator totals. Use tolerances large enough for measured hardware variance.
- Run a cheap smoke benchmark on stable CI hardware for every change; reserve noisy or expensive GPU trace jobs for scheduled builds, release candidates, or regressions. A failed required budget blocks promotion, while an advisory signal opens an investigation.
- Retain compact JSON summaries longer than raw traces. Encrypt trace artifacts, restrict access, set short retention, and redact or avoid tensor contents, input labels, paths, hostnames, request identifiers, and stack metadata that can expose source or customer data.
- Export summary metrics and capture metadata to the observability backend with code/model version, hardware, driver, profiler configuration, and trace URI. Correlate a sampled trace with deployment and request IDs, but keep high-cardinality or sensitive fields out of metric labels.

The cells below reuse this notebook's `torch.profiler` and `record_function` patterns. They are deliberately disabled by `RUN_PRODUCTION_* = False`; defining the helpers performs no workload, writes no artifact, and makes no cloud call.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable
import json
import platform


@dataclass(frozen=True)
class ProductionProfileConfig:
    artifact_dir: Path = Path("artifacts/pytorch-profiler")
    baseline_path: Path = Path("artifacts/pytorch-profiler/baseline.json")
    skip_first_steps: int = 2
    wait_steps: int = 1
    warmup_steps: int = 1
    active_steps: int = 2
    repeat: int = 1
    record_shapes: bool = False
    profile_memory: bool = True
    with_stack: bool = False
    max_trace_bytes: int = 50 * 1024 * 1024
    max_p95_latency_regression_pct: float = 10.0
    max_peak_memory_regression_pct: float = 10.0

    @property
    def total_steps(self) -> int:
        scheduled = self.wait_steps + self.warmup_steps + self.active_steps
        return self.skip_first_steps + self.repeat * scheduled


RUN_PRODUCTION_PROFILE = False
RUN_PRODUCTION_REGRESSION_GATE = False
PRODUCTION_PROFILE_CONFIG = ProductionProfileConfig()

In [ ]:
def notebook_training_step(_: int) -> None:
    """One representative step using the model and batch defined above."""
    with record_function("train.zero_grad"):
        optimizer.zero_grad(set_to_none=True)
    with record_function("train.forward"):
        logits = model(x_batch)
        loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    with record_function("train.backward"):
        loss.backward()
    with record_function("train.optimizer"):
        optimizer.step()


def capture_production_profile(
    step_fn: Callable[[int], None],
    config: ProductionProfileConfig,
) -> tuple[object, dict]:
    """Capture one bounded scheduled window; no work occurs until this is called."""
    config.artifact_dir.mkdir(parents=True, exist_ok=True)
    trace_paths = []

    def export_trace(profiler: object) -> None:
        trace_path = config.artifact_dir / f"trace-{len(trace_paths):02d}.json"
        profiler.export_chrome_trace(str(trace_path))
        trace_size = trace_path.stat().st_size
        if trace_size > config.max_trace_bytes:
            trace_path.unlink(missing_ok=True)
            raise RuntimeError(
                f"Trace exceeded {config.max_trace_bytes} bytes and was removed: {trace_size} bytes"
            )
        trace_paths.append(trace_path)

    selected_activities = [ProfilerActivity.CPU]
    if HAS_GPU:
        selected_activities.append(ProfilerActivity.CUDA)

    step_times_ms = []
    schedule = torch.profiler.schedule(
        skip_first=config.skip_first_steps,
        wait=config.wait_steps,
        warmup=config.warmup_steps,
        active=config.active_steps,
        repeat=config.repeat,
    )
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()

    with profile(
        activities=selected_activities,
        schedule=schedule,
        on_trace_ready=export_trace,
        record_shapes=config.record_shapes,
        profile_memory=config.profile_memory,
        with_stack=config.with_stack,
    ) as profiler:
        for step_index in range(config.total_steps):
            started = time.perf_counter()
            step_fn(step_index)
            if HAS_GPU:
                torch.cuda.synchronize()
            if step_index >= config.skip_first_steps:
                step_times_ms.append((time.perf_counter() - started) * 1000)
            profiler.step()

    capture = {
        "step_times_ms": step_times_ms,
        "peak_memory_bytes": torch.cuda.max_memory_allocated() if HAS_GPU else None,
        "trace_paths": [str(path) for path in trace_paths],
    }
    return profiler, capture

In [ ]:
def build_production_profile_report(
    profiler: object,
    capture: dict,
    config: ProductionProfileConfig,
) -> dict:
    step_times = np.asarray(capture["step_times_ms"], dtype=float)
    if step_times.size == 0:
        raise ValueError("No post-warmup step timings were captured.")

    operator_rows = sorted(
        profiler.key_averages(),
        key=lambda event: event.self_cpu_time_total,
        reverse=True,
    )[:10]
    top_operators = []
    for event in operator_rows:
        device_time = getattr(
            event,
            "self_device_time_total",
            getattr(event, "self_cuda_time_total", 0.0),
        )
        top_operators.append(
            {
                "name": event.key,
                "calls": event.count,
                "self_cpu_time_us": event.self_cpu_time_total,
                "self_device_time_us": device_time,
            }
        )

    profile_config = asdict(config)
    profile_config["artifact_dir"] = str(config.artifact_dir)
    profile_config["baseline_path"] = str(config.baseline_path)
    return {
        "schema_version": 1,
        "environment": {
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "device_type": DEVICE.type,
            "device_name": torch.cuda.get_device_properties(0).name if HAS_GPU else "cpu",
        },
        "config": profile_config,
        "metrics": {
            "measured_steps": int(step_times.size),
            "p50_step_ms": float(np.percentile(step_times, 50)),
            "p95_step_ms": float(np.percentile(step_times, 95)),
            "mean_steps_per_second": float(1000.0 / step_times.mean()),
            "peak_memory_bytes": capture["peak_memory_bytes"],
        },
        "top_operators": top_operators,
        "trace_paths": capture["trace_paths"],
    }


def profile_regressions(candidate: dict, baseline: dict, config: ProductionProfileConfig) -> list[str]:
    failures = []
    candidate_metrics = candidate["metrics"]
    baseline_metrics = baseline["metrics"]

    latency_change = (
        candidate_metrics["p95_step_ms"] / baseline_metrics["p95_step_ms"] - 1.0
    ) * 100.0
    if latency_change > config.max_p95_latency_regression_pct:
        failures.append(
            f"p95 step latency regressed {latency_change:.1f}% "
            f"(budget {config.max_p95_latency_regression_pct:.1f}%)"
        )

    candidate_memory = candidate_metrics.get("peak_memory_bytes")
    baseline_memory = baseline_metrics.get("peak_memory_bytes")
    if candidate_memory is not None and baseline_memory:
        memory_change = (candidate_memory / baseline_memory - 1.0) * 100.0
        if memory_change > config.max_peak_memory_regression_pct:
            failures.append(
                f"peak memory regressed {memory_change:.1f}% "
                f"(budget {config.max_peak_memory_regression_pct:.1f}%)"
            )
    return failures


def write_profile_report(report: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
    temporary_path.replace(path)


production_profile_report = None
if RUN_PRODUCTION_PROFILE:
    production_profiler, production_capture = capture_production_profile(
        notebook_training_step,
        PRODUCTION_PROFILE_CONFIG,
    )
    production_profile_report = build_production_profile_report(
        production_profiler,
        production_capture,
        PRODUCTION_PROFILE_CONFIG,
    )
    current_report_path = PRODUCTION_PROFILE_CONFIG.artifact_dir / "current.json"
    write_profile_report(production_profile_report, current_report_path)
    print(json.dumps(production_profile_report["metrics"], indent=2))
else:
    print("Production profile disabled: RUN_PRODUCTION_PROFILE = False")

if RUN_PRODUCTION_REGRESSION_GATE:
    if production_profile_report is None:
        raise RuntimeError("Enable RUN_PRODUCTION_PROFILE before running the regression gate.")
    baseline_report = json.loads(
        PRODUCTION_PROFILE_CONFIG.baseline_path.read_text(encoding="utf-8")
    )
    regression_failures = profile_regressions(
        production_profile_report,
        baseline_report,
        PRODUCTION_PROFILE_CONFIG,
    )
    if regression_failures:
        raise AssertionError("Performance regression gate failed: " + "; ".join(regression_failures))
    print("Performance regression gate passed.")
else:
    print("Production regression gate disabled: RUN_PRODUCTION_REGRESSION_GATE = False")